# CN7 가스: 원인별 전문가 확인 조건
학습 정상95% 확인 기준. 숫자 기준을 시험 정답으로 맞추지 않지만 조합 자체는 사후 개발이다.


In [1]:
import nbformat
old=nbformat.read("notebooks/170_cn7_gas_persistent_relation_experts.ipynb",as_version=4)
oldcode="\n".join(c.source for c in old.cells if c.cell_type=="code")
exec(oldcode[:oldcode.index("def predict(parent,q):")])
exec(oldcode[oldcode.index("def stats(a,b):"):oldcode.index("records=[];choices=[]")])
def selected(file,model,rule):
 a=pd.read_csv(OUT/file,parse_dates=["TimeStamp"])
 a=a[(a.model==model)&(a.rule==rule)].set_index("TimeStamp").loc[d.TimeStamp]
 assert len(a)==606 and a.index.is_unique
 assert np.array_equal(a.actual.to_numpy(),y)
 return a
base=pd.read_csv(SOURCE/"cn7_165_gas_supervised_complementary_predictions.csv",parse_dates=["TimeStamp"]).set_index("TimeStamp").loc[d.TimeStamp]
base165=base.three_specialists_confirmed.to_numpy(int)
p170=selected("cn7_170_persistence_predictions.csv","persistent_abs4_ET","normal95")
p173=selected("cn7_173_transfer_predictions.csv","mechanism_joint8_LR","inner_normal0.99")
base173=base165|p170.prediction.to_numpy(int)|p173.prediction.to_numpy(int)
rf=selected("cn7_175_magnitude_predictions.csv","local_flow_absolute6_RF","normal99")
lr=selected("cn7_175_magnitude_predictions.csv","flow_absolute6_LR","normal99")
tail=selected("cn7_172_normal_joint_predictions.csv","cold_fast","inner_normal0.995")
# Fixed 95% confirmation gates. No search over numeric thresholds using evaluation labels.
speed_high=np.zeros(N,bool);meter_high=np.zeros(N,bool);audit=[];gate_records=[]
for held in pd.unique(blocks):
 q=np.flatnonzero(blocks==held)
 ctx=train_context(np.ones(N,bool),q)
 normal=np.flatnonzero(ctx&d.any_fail.eq(0).to_numpy())
 for run in pd.unique(g[q]):
  ids=q[g[q]==run];ni=normal[g[normal]==run]
  if len(ni)<8:ni=normal
  assert set(ni).isdisjoint(q) and not ctx[q].any()
  speed_cut=float(np.quantile(d.Max_Injection_Speed_raw.iloc[ni],.95,method="higher"))
  meter_cut=float(np.quantile(d.Plasticizing_Position_raw.iloc[ni],.95,method="higher"))
  speed_high[ids]=d.Max_Injection_Speed_raw.iloc[ids].to_numpy()>speed_cut
  meter_high[ids]=d.Plasticizing_Position_raw.iloc[ids].to_numpy()>meter_cut
  audit.append({"heldout_block":held,"normal_reference":len(ni),"timestamp_overlap":len(set(d.TimeStamp.iloc[ni])&set(d.TimeStamp.iloc[q])),
  "test_raw_in_normal_reference":int(ctx[q].sum()),"speed_cut":speed_cut,"meter_cut":meter_cut})
  for i in ids:gate_records.append({"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,
   "speed_cut":speed_cut,"meter_cut":meter_cut,"speed_high":int(speed_high[i]),"meter_high":int(meter_high[i])})
arms={"localRF_fast":rf.prediction.to_numpy(bool)&speed_high,
 "absoluteLR_meter":lr.prediction.to_numpy(bool)&meter_high,
 "coldTail_fast":tail.prediction.to_numpy(bool)&speed_high}
policies={"base173":base173,
 "unconfirmed_all":base173|rf.prediction.to_numpy(int)|lr.prediction.to_numpy(int)|tail.prediction.to_numpy(int)}
for name,arm in arms.items():policies["base_plus_"+name]=base173|arm.astype(int)
policies["confirmed_RF_LR"]=base173|arms["localRF_fast"].astype(int)|arms["absoluteLR_meter"].astype(int)
policies["confirmed_all"]=policies["confirmed_RF_LR"]|arms["coldTail_fast"].astype(int)
rows=[]
for policy,p in policies.items():
 for cohort,mask in [("normal_and_gas",eligible),("all_official",np.ones(N,bool))]:
  rows.append({"policy":policy,"cohort":cohort,"N":int(mask.sum()),"gas":int(y[mask].sum()),**stats(y[mask],p[mask])})
mm=pd.DataFrame(rows)
cases=d[["TimeStamp","gas","any_fail","run_id","seq","time_block","Max_Injection_Speed_raw","Plasticizing_Position_raw"]].copy()
cases=cases.merge(pd.DataFrame(gate_records),on="TimeStamp",validate="one_to_one")
for name,v in policies.items():cases[name]=v
for name,v in arms.items():cases[name]=v.astype(int)
cases["raw_RF"]=rf.prediction.to_numpy(int)
cases["raw_LR"]=lr.prediction.to_numpy(int)
cases["raw_coldTail"]=tail.prediction.to_numpy(int)
cases.to_csv(OUT/"cn7_176_confirmed_expert_predictions.csv",index=False,encoding="utf-8-sig")
mm.to_csv(OUT/"cn7_176_confirmed_expert_metrics.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(audit).to_csv(OUT/"cn7_176_confirmation_audit.csv",index=False)
assert pd.DataFrame(audit).timestamp_overlap.eq(0).all()
assert pd.DataFrame(audit).test_raw_in_normal_reference.eq(0).all()
print("\nMETRICS\n",mm.to_string(index=False))
print("\nALL GAS\n",cases[cases.gas][["TimeStamp","base173","confirmed_all","localRF_fast","absoluteLR_meter","coldTail_fast"]].to_string(index=False))
print("\nNEW ALARMS\n",cases[cases.base173.ne(cases.confirmed_all)].to_string(index=False))
print("\nMISSED\n",cases[cases.gas&cases.confirmed_all.eq(0)].to_string(index=False))
lines=["# CN7 가스: 여러 원인 전문가의 방향 확인 앙상블","",
 "정상592/가스13=605샷, 같은20 시간블록/purge10. 기준173을 고정한다.",
 "175 조건부 충전편차 RF에 속도 상위5% 확인, 175 절대 충전편차 LR에 가소화 위치 상위5% 확인, 172 냉각+속도 정상통계 전문가에 속도 상위5% 확인을 붙였다.",
 "각 확인 기준은 해당 평가 구간과 앞뒤10샷을 뺀 같은Run의 학습 정상95백분위다. 기준보다 엄격히 커야 한다.",
 "속도·계량 확인은 이전 미탐/오탐을 보고 만든 사후 개발 가설이며 가스 원인을 입증하지 않는다.",
 "95%는 이번 확인 실험 전에 고정했고 숫자 임계값을 평가 정답으로 최적화하지 않았다.",
 "분류기·정상 기반 전문가·기존 조합 자체를 이미 본 평가 결과에서 골랐으므로 새 Run 독립 성능이 아니다.",
 "미성형까지 포함한 전체606샷을 별도로 기록한다. 정상 통계 전문가가 포함돼 전체를 순수 지도 모델이라고 부르면 안 된다.","",
 "## 전체 비교",mm.to_string(index=False),"",
 "## 가스13개",cases[cases.gas][["TimeStamp","base173","confirmed_all","localRF_fast","absoluteLR_meter","coldTail_fast"]].to_string(index=False),"",
 "## 새 경보",cases[cases.base173.ne(cases.confirmed_all)].to_string(index=False),"",
 "## 미탐",cases[cases.gas&cases.confirmed_all.eq(0)].to_string(index=False)]
(ROOT/"reports/cn7_176_confirmed_experts_review.md").write_text("\n".join(lines),encoding="utf-8")
(OUT/"cn7_176_confirmed_experts_provenance.json").write_text(json.dumps({
 "input_sha256":hashlib.sha256(INPUT.read_bytes()).hexdigest(),"validation":"same20/purge10",
 "classifiers":"existing nested OOF heads from175; frozen173 baseline",
 "statistical_arm":"172 cold_fast inner_normal.995",
 "confirmations":"strictly above same-run fold reference normal95 speed/meter",
 "caveat":"post-hoc development hypotheses/combination, not untouched independent validation",
 "no_eval_label_threshold_optimization":True},ensure_ascii=False,indent=2),encoding="utf-8")



METRICS
                     policy         cohort   N  gas  TP  FP  FN  TN  precision   recall       F1
                   base173 normal_and_gas 605   13   8   5   5 587   0.615385 0.615385 0.615385
                   base173   all_official 606   13   8   6   5 587   0.571429 0.615385 0.592593
           unconfirmed_all normal_and_gas 605   13  11  12   2 580   0.478261 0.846154 0.611111
           unconfirmed_all   all_official 606   13  11  13   2 580   0.458333 0.846154 0.594595
    base_plus_localRF_fast normal_and_gas 605   13   9   5   4 587   0.642857 0.692308 0.666667
    base_plus_localRF_fast   all_official 606   13   9   6   4 587   0.600000 0.692308 0.642857
base_plus_absoluteLR_meter normal_and_gas 605   13   8   5   5 587   0.615385 0.615385 0.615385
base_plus_absoluteLR_meter   all_official 606   13   8   6   5 587   0.571429 0.615385 0.592593
   base_plus_coldTail_fast normal_and_gas 605   13   9   8   4 584   0.529412 0.692308 0.600000
   base_plus_coldTail_fast   a

478